# Europa / Conference League Cycle Effect
Does playing Europa League or Conference League on a Thursday hurt a team's subsequent Premier League performance (typically Sun, 3 days later)?

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

DB_PATH = '../../infra/data/db/fotmob.db'
FIXTURES_PATH = '../../infra/data/json/fbref/fixtures_manual.csv'

## 1. Load Fixtures & ClubElo

In [2]:
fixtures = pd.read_csv(FIXTURES_PATH, parse_dates=['date'])
fixtures = fixtures.dropna(subset=['home_goals', 'away_goals']).sort_values('date').reset_index(drop=True)

conn = sqlite3.connect(DB_PATH)
elo_raw = pd.read_sql(
    "SELECT Club, Elo, [From] as elo_from FROM clubelo_ratings WHERE Country = 'ENG'",
    conn
)
conn.close()
elo_raw['elo_from'] = pd.to_datetime(elo_raw['elo_from'])
elo = (
    elo_raw
    .drop_duplicates(subset=['Club', 'elo_from'])
    .sort_values(['Club', 'elo_from'])
    .reset_index(drop=True)
)

base_map = pd.read_csv('../../infra/data/collectors/clubelo/clubelo_fbref_team_name_mapping.csv')
fbref_to_clubelo = dict(zip(base_map['fbref_name'], base_map['clubelo_name']))
fbref_to_clubelo.update({
    'Huddersfield Town': 'Huddersfield', 'Hull City': 'Hull',
    'Ipswich Town': 'Ipswich', 'Leicester City': 'Leicester',
    'Luton Town': 'Luton', 'Manchester City': 'Man City',
    'Manchester Utd': 'Man United', 'Newcastle United': 'Newcastle',
    "Nott'ham Forest": 'Forest', 'Nottingham Forest': 'Forest',
    'Norwich City': 'Norwich', 'Sheffield United': 'Sheffield United',
    'Stoke City': 'Stoke', 'Swansea City': 'Swansea',
    'Tottenham Hotspur': 'Tottenham', 'West Ham United': 'West Ham',
    'Cardiff City': 'Cardiff',
})

print(f"Fixtures loaded: {len(fixtures)} | ClubElo rows: {len(elo)}")

Fixtures loaded: 8626 | ClubElo rows: 14521


## 2. Flag Europa/Conference League Thursday Appearances

For each team, find every Thursday Europa League or Conference League match, then identify the next PL match. Flag it as `played_thu` if that PL match falls within 4 days (almost always Thu→Sun = 3 days).

In [3]:
# All EL / Conference League Thursday appearances
euro_thu = fixtures[fixtures['league_name'].isin(['Europa League', 'Conference League'])].copy()
euro_thu = euro_thu[euro_thu['date'].dt.day_name() == 'Thursday']

euro_home = euro_thu[['date', 'home_team']].rename(columns={'home_team': 'team'})
euro_away = euro_thu[['date', 'away_team']].rename(columns={'away_team': 'team'})
euro_thu_teams = pd.concat([euro_home, euro_away]).sort_values(['team', 'date']).reset_index(drop=True)
euro_thu_teams['euro_thu_date'] = euro_thu_teams['date']

print(f"EL/Conference Thu appearances: {len(euro_thu_teams)}")
print(euro_thu_teams['team'].value_counts().head(10))

EL/Conference Thu appearances: 4462
team
Roma                   83
Rangers                66
Ludogorets Razgrad     65
Slavia Prague          63
AZ Alkmaar             61
Braga                  60
Real Betis             57
Arsenal                56
Eintracht Frankfurt    53
Olympiacos             53
Name: count, dtype: int64


In [4]:
# All PL appearances per team
pl = fixtures[fixtures['league_name'] == 'Premier League'].copy().reset_index(drop=True)
pl_home = pl[['date', 'home_team']].rename(columns={'home_team': 'team'})
pl_away = pl[['date', 'away_team']].rename(columns={'away_team': 'team'})
pl_teams = pd.concat([pl_home, pl_away]).sort_values(['team', 'date']).reset_index(drop=True)

# For each EL Thursday, find the next PL match for that team
next_pl = pd.merge_asof(
    euro_thu_teams[['team', 'euro_thu_date']].sort_values('euro_thu_date'),
    pl_teams.rename(columns={'date': 'pl_date'}).sort_values('pl_date'),
    left_on='euro_thu_date',
    right_on='pl_date',
    by='team',
    direction='forward'
)
next_pl['days_to_pl'] = (next_pl['pl_date'] - next_pl['euro_thu_date']).dt.days

print("Days from EL Thursday to next PL match:")
print(next_pl['days_to_pl'].value_counts().sort_index().head(10))

# Flag as Thu/Sun cycle if PL match is within 4 days
thu_sun_flags = next_pl[next_pl['days_to_pl'] <= 4][['team', 'pl_date']].drop_duplicates()
thu_sun_flags['played_thu'] = True
print(f"\nThu/Sun PL match-team instances: {len(thu_sun_flags)}")

Days from EL Thursday to next PL match:
days_to_pl
2.0       1
3.0     250
4.0       7
5.0       1
6.0       1
7.0       3
9.0       1
10.0      6
13.0      1
14.0      1
Name: count, dtype: int64

Thu/Sun PL match-team instances: 258


## 3. Build PL Team-Match Dataset

In [5]:
def lookup_elo_asof(df, team_col, date_col, elo_df):
    results = []
    for club, group in df.groupby(team_col, sort=False):
        club_elo = elo_df[elo_df['Club'] == club][['elo_from', 'Elo']].sort_values('elo_from')
        if club_elo.empty:
            results.append(pd.Series(np.nan, index=group.index))
            continue
        matched = pd.merge_asof(
            group[[date_col]].sort_values(date_col), club_elo,
            left_on=date_col, right_on='elo_from', direction='backward'
        )
        matched.index = group.sort_values(date_col).index
        results.append(matched['Elo'])
    return pd.concat(results).reindex(df.index)


# Reshape PL to one row per team per match
home = pl.assign(team=pl['home_team'], opponent=pl['away_team'],
                 goals_for=pl['home_goals'], goals_against=pl['away_goals'], is_home=1)
away = pl.assign(team=pl['away_team'], opponent=pl['home_team'],
                 goals_for=pl['away_goals'], goals_against=pl['home_goals'], is_home=0)
tm = pd.concat([home, away], ignore_index=True)
tm['goal_diff'] = tm['goals_for'] - tm['goals_against']

# Join elo
tm['team_elo_name']     = tm['team'].map(fbref_to_clubelo)
tm['opponent_elo_name'] = tm['opponent'].map(fbref_to_clubelo)
tm['team_elo']          = lookup_elo_asof(tm, 'team_elo_name', 'date', elo)
tm['opponent_elo']      = lookup_elo_asof(tm, 'opponent_elo_name', 'date', elo)

# Join Thu/Sun flags for team and opponent
tm = tm.merge(
    thu_sun_flags.rename(columns={'team': 'team', 'pl_date': 'date'}),
    on=['team', 'date'], how='left'
)
tm = tm.merge(
    thu_sun_flags.rename(columns={'team': 'opponent', 'pl_date': 'date', 'played_thu': 'opp_played_thu'}),
    on=['opponent', 'date'], how='left'
)
tm['played_thu']     = tm['played_thu'].fillna(False)
tm['opp_played_thu'] = tm['opp_played_thu'].fillna(False)

print(f"Total team-match rows: {len(tm)}")
print(f"Rows where team played Thu: {tm['played_thu'].sum()}")
print(f"Rows where opp played Thu:  {tm['opp_played_thu'].sum()}")

Total team-match rows: 6718
Rows where team played Thu: 258
Rows where opp played Thu:  258


## 4. Raw Means — Does Thu/Sun Hurt?

In [6]:
print("Mean goal_diff by played_thu (team):")
print(tm.groupby('played_thu')['goal_diff'].agg(['mean', 'count']))
print()
print("Mean goal_diff by opp_played_thu (opponent):")
print(tm.groupby('opp_played_thu')['goal_diff'].agg(['mean', 'count']))
print()
print("2x2 cross-tab (team played_thu vs opp_played_thu):")
print(tm.groupby(['played_thu', 'opp_played_thu'])['goal_diff'].agg(['mean', 'count']).round(3))

Mean goal_diff by played_thu (team):
                mean  count
played_thu                 
False      -0.013777   6460
True        0.344961    258

Mean goal_diff by opp_played_thu (opponent):
                    mean  count
opp_played_thu                 
False           0.013777   6460
True           -0.344961    258

2x2 cross-tab (team played_thu vs opp_played_thu):
                            mean  count
played_thu opp_played_thu              
False      False           0.000   6224
           True           -0.377    236
True       False           0.377    236
           True            0.000     22


## 5. OLS Model — Europa Cycle Effect

In [7]:
model_df = tm[['goal_diff', 'is_home', 'team_elo', 'opponent_elo', 'played_thu', 'opp_played_thu']].dropna().copy()

print(f"Fitting on {len(model_df)} rows")
formula = 'goal_diff ~ is_home + team_elo + opponent_elo + played_thu + opp_played_thu'
result = smf.ols(formula, data=model_df).fit()
print(result.summary())

Fitting on 6718 rows
                            OLS Regression Results                            
Dep. Variable:              goal_diff   R-squared:                       0.224
Model:                            OLS   Adj. R-squared:                  0.223
Method:                 Least Squares   F-statistic:                     387.5
Date:                Sat, 18 Apr 2026   Prob (F-statistic):               0.00
Time:                        05:16:13   Log-Likelihood:                -13082.
No. Observations:                6718   AIC:                         2.618e+04
Df Residuals:                    6712   BIC:                         2.622e+04
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Interce

## 6. Subset: European Teams Only

Limit to teams that actually participated in European competition that season — controls for the selection effect (European teams are stronger and face each other more).

In [8]:
# Teams that appeared in EL/Conference at any point in each season
euro_teams_by_season = (
    euro_thu_teams
    .merge(fixtures[['date', 'season']].drop_duplicates(), on='date', how='left')
    .groupby('season')['team']
    .apply(set)
)

def is_european(row):
    season_teams = euro_teams_by_season.get(row['season'], set())
    return row['team'] in season_teams or row['opponent'] in season_teams

tm['either_european'] = tm.apply(is_european, axis=1)
euro_subset = tm[tm['either_european']].copy()

print(f"Rows involving at least one European team: {len(euro_subset)}")
print(f"  played_thu: {euro_subset['played_thu'].sum()} | opp_played_thu: {euro_subset['opp_played_thu'].sum()}")
print()

model_df_e = euro_subset[['goal_diff','is_home','team_elo','opponent_elo','played_thu','opp_played_thu']].dropna()
result_e = smf.ols('goal_diff ~ is_home + team_elo + opponent_elo + played_thu + opp_played_thu', data=model_df_e).fit()
print(result_e.summary())

Rows involving at least one European team: 1898
  played_thu: 258 | opp_played_thu: 258

                            OLS Regression Results                            
Dep. Variable:              goal_diff   R-squared:                       0.203
Model:                            OLS   Adj. R-squared:                  0.201
Method:                 Least Squares   F-statistic:                     96.19
Date:                Sat, 18 Apr 2026   Prob (F-statistic):           1.79e-90
Time:                        05:16:13   Log-Likelihood:                -3739.0
No. Observations:                1898   AIC:                             7490.
Df Residuals:                    1892   BIC:                             7523.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------